In [1]:
import os
import sys

os.chdir("..")
sys.path.append("src")

In [2]:
import torch
import pandas as pd
import matplotlib.pyplot as plt

from wildfire_gnn.utils.config import load_yaml_config
from wildfire_gnn.pipelines.gnn_pipeline import GNNPipeline
from wildfire_gnn.features.feature_engineering import (
    add_degree_feature,
    add_neighborhood_summary_features,
)
from wildfire_gnn.data.graph_splitters import (
    attach_masks_from_split_file,
    print_mask_summary,
)

In [3]:
config = load_yaml_config("configs/gnn_config.yaml")
pipeline = GNNPipeline(config)

data = torch.load(
    config["paths"]["graph_data_path"],
    map_location="cpu",
    weights_only=False
)

data = attach_masks_from_split_file(
    data,
    config["paths"]["spatial_split_path"]
)

print(data)
print("x shape:", data.x.shape)
print("y shape:", data.y.shape)
print("edge_index shape:", data.edge_index.shape)
print_mask_summary(data)

Data(x=[300000, 7], edge_index=[2, 991684], y=[300000, 1], pos=[300000, 2], num_nodes_original_grid=3589100, num_valid_nodes_before_sampling=702972, num_valid_nodes=300000, reference_height=1900, reference_width=1889, target_name='Burn_Prob.img', feature_names=[7], train_mask=[300000], val_mask=[300000], test_mask=[300000])
x shape: torch.Size([300000, 7])
y shape: torch.Size([300000, 1])
edge_index shape: torch.Size([2, 991684])
Mask summary:
train: 199167
val  : 40718
test : 60115
total: 300000
nodes: 300000


In [4]:
# Save raw target for weighting + final evaluation
data.y_raw = data.y.clone()

# Apply training-time target transform
if config["data"].get("target_transform", "none") == "log1p":
    data.y = torch.log1p(data.y)

print("Target stats after training transform:")
print("min:", float(data.y.min()))
print("max:", float(data.y.max()))
print("raw max:", float(data.y_raw.max()))

Target stats after training transform:
min: 2.3405691536027007e-05
max: 0.2225854992866516
raw max: 0.24930262565612793


In [5]:
if config["feature_engineering"].get("add_degree_feature", False):
    data = add_degree_feature(data)

if config["feature_engineering"].get("add_neighborhood_features", False):
    aggs = set(config["feature_engineering"].get("neighborhood_aggs", []))
    data = add_neighborhood_summary_features(
        data,
        add_mean=("mean" in aggs),
        add_std=("std" in aggs),
        add_max=("max" in aggs),
        add_residual=("residual" in aggs),
    )

print("Engineered x shape:", data.x.shape)

Engineered x shape: torch.Size([300000, 40])


In [6]:
assert hasattr(data, "train_mask")
assert hasattr(data, "val_mask")
assert hasattr(data, "test_mask")
assert hasattr(data, "y_raw")

assert int(data.train_mask.sum() + data.val_mask.sum() + data.test_mask.sum()) == data.num_nodes
print("All graph masks and targets are ready.")

All graph masks and targets are ready.


In [7]:
train_outputs = pipeline.train(data, stage="stage1")
train_outputs.history.tail()

,epoch,train_loss,val_loss
51,52,0.000879,0.000851
52,53,0.000876,0.000850
53,54,0.000875,0.000846
54,55,0.000874,0.000839
55,56,0.000872,0.000832
